In [1]:
import os, json
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [3]:
import truststore
truststore.inject_into_ssl()

#This is optional. I use VPN in my computer. Why I have to use this

from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

MODEL  = "gpt-5-nano"  

OpenAI client ready.


# Homework - Open Mateo Get Weather

In [ ]:
# Define a get_weather tool
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Same question as before — but now the model has a tool!
response = client.responses.create(
    model=MODEL,
    input="What's the weather in Paris right now?",
    tools=[weather_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: get_weather
  Arguments: {"location":"Paris"}
  Call ID: call_FDJ7OdLwosbaVElvkjcd3xyU
 Response:


In [9]:
# Define a get_weather tool
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Our fake weather function (in production, this would call a real API)
def get_weather(location):
    fake_data = {"Tel Aviv": "28°C, sunny", "Paris": "18°C, cloudy", "London": "14°C, rain"}
    return fake_data.get(location, f"No data for {location}")

# Step 1: Ask with the weather tool
response = client.responses.create(
    model=MODEL,
    input="What's the weather like in Tel Aviv and London?",
    tools=[weather_tool],
)

# Step 2 & 3: Find all function calls, execute each
print("Model requested these calls:")
new_input = list(response.output)

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        
        # Step 4: Append our result
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })

# Step 5: Model composes final answer with real data
final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
)
print(f"\nFinal answer:\n{final.output_text}")

Model requested these calls:
  → get_weather({'location': 'Tel Aviv'}) = 28°C, sunny
  → get_weather({'location': 'London'}) = 14°C, rain

Final answer:
Here are the current conditions:

- Tel Aviv: 28°C, sunny
- London: 14°C, rain

Want an hourly forecast or a 7-day outlook for either city, or set up weather alerts?


In [10]:
# let's design the open mateo application

import requests

# VPN-friendly session: ignore HTTP(S)_PROXY / NO_PROXY env vars
SESSION = requests.Session()
SESSION.trust_env = False

GEOCODE_URL  = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# WMO weather codes -> human label
WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "light rain", 63: "moderate rain", 65: "heavy rain",
    71: "light snow", 73: "moderate snow", 75: "heavy snow",
    80: "rain showers", 81: "heavy rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm w/ hail", 99: "severe thunderstorm w/ hail",
}

def geocode(location: str):
    """City name -> (lat, lon, pretty_name). This is how 'Paris' becomes coordinates."""
    r = SESSION.get(
        GEOCODE_URL,
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=10,
    )
    r.raise_for_status()
    results = r.json().get("results") or []
    if not results:
        return None
    place = results[0]
    pretty = ", ".join(filter(None, [place.get("name"), place.get("admin1"), place.get("country")]))
    return place["latitude"], place["longitude"], pretty



geocode("Tel Aviv")


(32.08088, 34.78057, 'Tel Aviv, Tel Aviv, Israel')

In [11]:

# Real get_weather — talks to Open-Meteo (free, no API key needed)
def get_weather(location: str) -> str:
    # Step 1: geocode the city name into lat/lon
    geo = geocode(location)
    if geo is None:
        return f"No data for {location}"
    lat, lon, resolved = geo
    print(f"  [geocoded] {location!r} -> ({lat}, {lon}) = {resolved}")

    # Step 2: ask Open-Meteo for current weather at those coordinates
    r = SESSION.get(
        FORECAST_URL,
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "timezone": "auto",
        },
        timeout=10,
    )
    r.raise_for_status()
    data = r.json()
    cur   = data.get("current", {})
    units = data.get("current_units", {})

    condition = WEATHER_CODES.get(cur.get("weather_code"), "unknown")
    return (
        f"{resolved}: {cur.get('temperature_2m')}{units.get('temperature_2m', '°C')}, "
        f"{condition}, humidity {cur.get('relative_humidity_2m')}{units.get('relative_humidity_2m', '%')}, "
        f"wind {cur.get('wind_speed_10m')} {units.get('wind_speed_10m', 'km/h')}"
    )


print(get_weather("Bengaluru"))
print(get_weather("Tel Aviv"))
print(get_weather("London"))
print(get_weather("Ahmedabad"))

  [geocoded] 'Bengaluru' -> (12.97194, 77.59369) = Bengaluru, Karnataka, India
Bengaluru, Karnataka, India: 22.4°C, overcast, humidity 96%, wind 11.7 km/h
  [geocoded] 'Tel Aviv' -> (32.08088, 34.78057) = Tel Aviv, Tel Aviv, Israel
Tel Aviv, Tel Aviv, Israel: 26.5°C, clear sky, humidity 50%, wind 12.4 km/h
  [geocoded] 'London' -> (51.50853, -0.12574) = London, England, United Kingdom
London, England, United Kingdom: 17.4°C, light drizzle, humidity 69%, wind 19.8 km/h
  [geocoded] 'Ahmedabad' -> (23.02579, 72.58727) = Ahmedabad, Gujarat, India
Ahmedabad, Gujarat, India: 34.3°C, thunderstorm, humidity 55%, wind 12.6 km/h


In [12]:
# Same flow as before, but now backed by a real API.
# weather_tool (the schema) is reused as-is — only get_weather changed.

response = client.responses.create(
    model=MODEL,
    input="What's the weather like in Tel Aviv and London right now?",
    tools=[weather_tool],
)

new_input = list(response.output)

print("Model requested these calls:")
for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })

final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
)
print(f"\nFinal answer:\n{final.output_text}")

Model requested these calls:
  [geocoded] 'Tel Aviv' -> (32.08088, 34.78057) = Tel Aviv, Tel Aviv, Israel
  → get_weather({'location': 'Tel Aviv'}) = Tel Aviv, Tel Aviv, Israel: 26.5°C, clear sky, humidity 50%, wind 12.4 km/h
  [geocoded] 'London' -> (51.50853, -0.12574) = London, England, United Kingdom
  → get_weather({'location': 'London'}) = London, England, United Kingdom: 17.4°C, light drizzle, humidity 69%, wind 19.8 km/h

Final answer:
Here are the current conditions:

- Tel Aviv: 26.5°C, clear sky. Humidity 50%, wind 12.4 km/h.
- London: 17.4°C, light drizzle. Humidity 69%, wind 19.8 km/h.

Would you like a forecast for the next few days or switch to Fahrenheit?


# Evaluate Your GENAI Application

There are some tests:
1. That you want to do on every commit, as part of your CI/CD pipeline. These are called **Unit Tests**.
2. Some tests are meant to be done on golden set. 

## The Story

> *Imagine you've built a customer support chatbot for **TechMart**, an online electronics store. It answers 1,000 questions a day. Your boss asks: "How good is it?"*
>
> *You check accuracy — but against what? There's no single right answer to "Can I return a partially used product?" The answer depends on tone, policy nuance, completeness, and empathy.*
>

In [13]:

# Our scenario: TechMart customer support chatbot
question = "What's your return policy for electronics?"

expected_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "Opened software and digital downloads are non-refundable. "
    "For defective items, we offer a 90-day exchange warranty."
)

# Let's generate a chatbot response using OpenAI
response = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "system", "content": (
            "You are a helpful customer support agent for TechMart electronics store. "
            "TechMart's return policy: 30-day returns for electronics in original packaging "
            "with accessories. Opened software/digital downloads non-refundable. "
            "90-day exchange warranty for defective items."
        )},
        {"role": "user", "content": question}
    ]
)

actual_answer = response.output_text
print("📋 QUESTION:", question)
print()
print("✅ EXPECTED ANSWER:")
print(expected_answer)
print()
print("🤖 CHATBOT ANSWER:")
print(actual_answer)

📋 QUESTION: What's your return policy for electronics?

✅ EXPECTED ANSWER:
You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. Opened software and digital downloads are non-refundable. For defective items, we offer a 90-day exchange warranty.

🤖 CHATBOT ANSWER:
Here’s our electronics return policy:

- 30-day returns: You can return electronics within 30 days of purchase as long as the item is in its original packaging with all included accessories.
- Opened software/digital downloads: Not refundable.
- Defective items: We offer a 90-day exchange warranty for items that are defective. If defective, we’ll exchange the item.

If you’d like, share your order number and the item, and I can guide you through the return or warranty process.


In [14]:
misleading_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "We also offer FREE LIFETIME WARRANTY on everything and PRICE MATCHING "
    "against any competitor!"  
)

1. Heuristic / Statistical.
2. Human evaluation or LLM as a judge
3. Golden set evaluation.

In [15]:

# Pillar 1: Heuristic / Code-Based Evaluation

# These are simple but catch real problems in production!

print("Actual answer: ", actual_answer)

def evaluate_heuristics(response: str) -> dict:
    """Basic code-based checks for a customer support chatbot."""
    checks = {}

    # Length check — too short = probably unhelpful, too long = overwhelming
    word_count = len(response.split())
    checks["appropriate_length"] = 20 <= word_count <= 300
    checks["word_count"] = word_count

    # Contains required elements
    checks["has_greeting_or_direct_answer"] = not response.startswith("I don't")
    checks["no_competitor_mentions"] = not any(
        comp in response.lower() for comp in ["amazon", "bestbuy", "best buy", "walmart"]
    )

    # Safety checks
    checks["no_profanity"] = not any(
        word in response.lower() for word in ["damn", "hell", "stupid"]
    )

    # Format check — should not contain raw code or system prompts
    checks["no_system_prompt_leak"] = "system:" not in response.lower()
    checks["no_raw_json"] = not response.strip().startswith("{")

    return checks

# Test on our chatbot's response
print("🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:")
print("=" * 50)
results = evaluate_heuristics(actual_answer)
for check, passed in results.items():
    status = "✅" if (passed if isinstance(passed, bool) else True) else "❌"
    print(f"  {status} {check}: {passed}")

print()
print("💡 These checks are fast and deterministic — perfect for CI/CD.")
print("   But they can't tell you if the answer is ACTUALLY CORRECT or HELPFUL.")

Actual answer:  Here’s our electronics return policy:

- 30-day returns: You can return electronics within 30 days of purchase as long as the item is in its original packaging with all included accessories.
- Opened software/digital downloads: Not refundable.
- Defective items: We offer a 90-day exchange warranty for items that are defective. If defective, we’ll exchange the item.

If you’d like, share your order number and the item, and I can guide you through the return or warranty process.
🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:
  ✅ appropriate_length: True
  ✅ word_count: 78
  ✅ has_greeting_or_direct_answer: True
  ✅ no_competitor_mentions: True
  ✅ no_profanity: True
  ✅ no_system_prompt_leak: True
  ✅ no_raw_json: True

💡 These checks are fast and deterministic — perfect for CI/CD.
   But they can't tell you if the answer is ACTUALLY CORRECT or HELPFUL.


## LLM As a Judge

In [16]:

# Pillar 2: LLM-as-a-Judge — Build one from scratch!

# Before we use frameworks, let's understand what's happening under the hood.

def llm_judge(question, response, criteria, model="gpt-5-nano"):
    """A simple LLM-as-a-Judge implementation from scratch."""

    judge_prompt = f"""You are an expert evaluator for a customer support chatbot.

    Evaluate the following response on this criteria: {criteria}

    USER QUESTION: {question}
    CHATBOT RESPONSE: {response}

    Score from 1-5 where:
    1 = Completely fails the criteria
    2 = Mostly fails with minor positives
    3 = Partially meets criteria
    4 = Mostly meets criteria with minor issues
    5 = Fully meets criteria

    Respond in this exact JSON format:
    {{"score": <int>, "reason": "<brief explanation>"}}"""

    result = client.responses.create(
        model=model,
        input=[{"role": "user", "content": judge_prompt}]
    )

    try:
        # Parse JSON from response
        text = result.output_text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text)
    except:
        return {"score": 0, "reason": f"Failed to parse: {result.output_text[:200]}"}

# ----- Evaluate on multiple criteria -----
criteria_list = {
    "Accuracy": "Is the response factually correct based on TechMart's return policy?",
    "Helpfulness": "Does the response fully address the user's question in a helpful way?",
    "Tone": "Is the tone professional, friendly, and empathetic?",
    "Completeness": "Does the response cover all relevant aspects (timeframe, conditions, exceptions)?"
}

print("🧑‍⚖️ LLM-AS-A-JUDGE EVALUATION")
print("=" * 60)
print(f"Question: {question}")
print(f"Response: {actual_answer[:150]}...")
print()

scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, actual_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")

🧑‍⚖️ LLM-AS-A-JUDGE EVALUATION
Question: What's your return policy for electronics?
Response: Here’s our electronics return policy:

- 30-day returns: You can return electronics within 30 days of purchase as long as the item is in its original ...

  Accuracy: ⭐⭐☆☆☆ (2/5)
    → Cannot confirm TechMart's official return policy from the given information. The answer asserts specific terms (30-day electronics return with original packaging, 90-day defect exchange, non-refundable opened software/digital) that may not align with TechMart's policy. Without the actual policy, accuracy cannot be determined.

  Helpfulness: ⭐⭐⭐⭐☆ (4/5)
    → Covers the key electronics return terms (30-day return in original packaging with accessories; non-refundable opened software/digital; 90-day exchange warranty for defects) and offers assistance with the process. However, it lacks details on refund method, who pays return shipping, restocking fees, and a clear step-by-step return/warranty procedure.

  Tone

In [17]:
scores


{'Accuracy': {'score': 2,
  'reason': "Cannot confirm TechMart's official return policy from the given information. The answer asserts specific terms (30-day electronics return with original packaging, 90-day defect exchange, non-refundable opened software/digital) that may not align with TechMart's policy. Without the actual policy, accuracy cannot be determined."},
 'Helpfulness': {'score': 4,
  'reason': 'Covers the key electronics return terms (30-day return in original packaging with accessories; non-refundable opened software/digital; 90-day exchange warranty for defects) and offers assistance with the process. However, it lacks details on refund method, who pays return shipping, restocking fees, and a clear step-by-step return/warranty procedure.'},
 'Tone': {'score': 4,
  'reason': 'The response is clear, professional, and offers help, giving policy details and inviting the user to share their order number. It reads as friendly and helpful. It could be more empathetic with an e

In [18]:
# ----- Now judge the HALLUCINATED response -----
print("🚨 JUDGING THE HALLUCINATED RESPONSE")
print("=" * 60)
print(f"Response: {misleading_answer[:150]}...")
print()

scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, misleading_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")


print("💡 KEY INSIGHT: The LLM judge CATCHES the hallucination that ROUGE missed!")
print("   This is why LLM-as-a-Judge is the dominant evaluation approach in 2025.")

🚨 JUDGING THE HALLUCINATED RESPONSE
Response: You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. We al...

  Accuracy: ⭐⭐☆☆☆ (2/5)
    → Includes an implausible 'FREE LIFETIME WARRANTY on everything' claim. Even if 30-day returns/original packaging exist, the major inaccuracy makes the answer not factually correct.

  Helpfulness: ⭐⭐⭐☆☆ (3/5)
    → Partially answers the question: it states a 30-day return window and packaging conditions but omits important return policy details (exclusions, proof of purchase, who pays return shipping, refund method, how to initiate a return) and mixes in unrelated offers (warranty, price matching) that aren't part of the return policy.

  Tone: ⭐⭐⭐☆☆ (3/5)
    → Clear and professional with concrete policy details, but it lacks warmth and empathy (no empathetic phrasing or acknowledgment of the customer’s situation).

  Completeness: ⭐⭐⭐☆☆ (3/5)
    → Covers ba

# DeepEval